##### ✅ *Models*

In [ ]:
# @title Installs (Minimal für Google GenAI, OpenAI SDK, LiteLLM & Model Garden)
############################################

# 1. Benötigte Pakete für Models installieren
%pip install --upgrade --quiet google-genai google-cloud-aiplatform openai litellm google-adk

# 2. Versionen kurz prüfen
%pip show google-genai google-cloud-aiplatform openai litellm

In [2]:
# @title Imports (Minimal für Models)
############################################

import os
import warnings
warnings.filterwarnings('ignore')

# 1. Google Cloud Auth & Vertex AI
import google.auth
import google.auth.transport.requests
import vertexai
from vertexai import model_garden

# 2. Google GenAI SDK
from google import genai
from google.genai import types

# 3. OpenAI SDK (für Vertex AI OpenAI-kompatiblen Endpunkt)
from openai import OpenAI

# 4. LiteLLM & ADK Wrapper
import litellm
from google.adk.models.lite_llm import LiteLlm

print("✅ Alle Imports erfolgreich geladen!")

✅ Alle Imports erfolgreich geladen!


In [3]:
# @title Environmental Variables & GCP Setup
############################################

PROJECT_ID = "deltorobarba"
LOCATION = "us-central1"

# These tell the underlying google-genai client and SDK to use Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

# Required by LiteLLM for Vertex AI partner models
os.environ["VERTEXAI_PROJECT"] = PROJECT_ID
os.environ["VERTEXAI_LOCATION"] = LOCATION

# Initialize Vertex AI SDK & Client via Application Default Credentials (ADC)
vertexai.init(project=PROJECT_ID, location=LOCATION)
client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

print("✅ Erfolgreich über Application Default Credentials (ADC) verbunden.")

✅ Erfolgreich über Application Default Credentials (ADC) verbunden.


In [11]:
# @title Google SDK

# https://github.com/se02035/local-llm-proxy it helps to spin up a litellm proxy with local ollama. also supports a public endpoint using ngrok

# Locations: https://docs.cloud.google.com/vertex-ai/generative-ai/docs/learn/locations
# Google Models: https://docs.cloud.google.com/vertex-ai/generative-ai/docs/models
# Versions: https://docs.cloud.google.com/vertex-ai/generative-ai/docs/learn/model-versions

# client = genai.Client(http_options=HttpOptions(api_version="v1")) --> only for Google models directly
# https://github.com/GoogleCloudPlatform/generative-ai/blob/main/sdk/intro_genai_sdk.ipynb
# https://docs.cloud.google.com/vertex-ai/generative-ai/docs/open-models/use-maas
from google import genai
from google.genai.types import GenerateContentConfig
from google.genai import types

PROJECT_ID="deltorobarba"
LOCATION = "global"
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

gemini    = "google/gemini-3.1-pro-preview"
gemma     = "google/gemma-4-26b-a4b-it-maas"
grok      = "xai/grok-4.1-fast-reasoning"
deepseek  = "deepseek-ai/deepseek-v3.2-maas"
kimi      = "moonshotai/kimi-k2-thinking-maas"
qwen      = "qwen/qwen3-next-80b-a3b-thinking-maas"
minimax   = "minimaxai/minimax-m2-maas"
glm       = "zai-org/glm-5-maas"
llama     = "meta/llama-3.3-70b-instruct-maas"   # LOCATION = "us-central1"
claude    = "anthropic/claude-opus-4-7"

system_instruction = """You are a helpful machine learning advisor and answer briefly."""
prompt = """Hi, which LLM are you?"""

generate_content_config = types.GenerateContentConfig(
    temperature=0.4,
    top_p=0.95,
    top_k=20,
    candidate_count=1,
    seed=5,
    max_output_tokens=60,
    stop_sequences=["STOP!"],
    presence_penalty=0.0,
    frequency_penalty=0.0,
    system_instruction=system_instruction,)

response = client.models.generate_content(
    model=gemma,                       # <--- ⚠️ Define model here
    contents=prompt,
    config=generate_content_config,
)
print("✅ Success")
print(response.text)

✅ Success
I am a large language model, trained by Google.


In [7]:
# @title OpenAI SDK
#################################################

from openai import OpenAI

PROJECT  = "deltorobarba"
LOCATION = "global"

# ⚠️ Accept User Agreements per Model!
grok = "xai/grok-4.1-fast-reasoning"        # ✅ https://docs.cloud.google.com/vertex-ai/generative-ai/docs/partner-models/grok/grok-4-1-fast
gemma = "google/gemma-4-26b-a4b-it-maas"    # ✅ https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/google/gemma-4-26b-a4b-it
gemini = "google/gemini-3.1-pro-preview"    # ✅ https://docs.cloud.google.com/vertex-ai/generative-ai/docs/models/gemini/3-1-pro
deepseek = "deepseek-ai/deepseek-v3.2-maas" # ✅ https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/deepseek/deepseek-v32

# Get access token from Application Default Credentials (run `gcloud auth application-default login` once beforehand)
credentials, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
credentials.refresh(google.auth.transport.requests.Request())

# Point OpenAI SDK at Vertex's OpenAI-compatible endpoint
client = OpenAI(
    base_url=f"https://aiplatform.googleapis.com/v1/projects/{PROJECT}/locations/{LOCATION}/endpoints/openapi",
    api_key=credentials.token,
)

# Call selected model like any OpenAI model
response = client.chat.completions.create(
    model=deepseek,
    messages=[{"role": "user", "content": "Hi, which LLM are you?"}],
    temperature=0.2,
)

print("✅ Success")
print(response.choices[0].message.content)

✅ Success
你好！我是DeepSeek，由深度求索公司创造的AI助手！😊

我是一个纯文本模型，具有以下特点：
- 完全免费使用
- 支持128K上下文长度
- 可以处理文件上传（图像、txt、pdf、ppt、word、excel等）
- 支持联网搜索（需要手动开启）
- 可通过官方应用商店下载App使用

虽然我不支持多模态识别功能，但我可以读取上传文件中的文字信息来帮助你。我的知识截止到2024年7月，会尽我所能为你提供准确、有用的回答！

有什么问题我可以帮你解答吗？无论是学习、工作还是生活方面的问题，我都很乐意协助你！✨


In [10]:
# @title LiteLLM (Chinesische LLMs 🇨🇳)
#################################################

# DeepSeek V3.2 - https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/deepseek/deepseek-v32
deepseek = LiteLlm(
    model="vertex_ai/deepseek-ai/deepseek-v3.2-maas",
    vertex_project="deltorobarba",
    vertex_location="global")

# Kimi K2 - https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/kimi/kimi-k2-thinking
kimi = LiteLlm(
    model="vertex_ai/kimi/kimi-k2-thinking-maas",
    vertex_project="deltorobarba",
    vertex_location="global")

# Qwen 3 - https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/qwen/qwen3-next-thinking
qwen = LiteLlm(
    model="vertex_ai/qwen/qwen3-next-80b-a3b-thinking-maas",
    vertex_project="deltorobarba",
    vertex_location="global")

# MiniMax - https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/minimax
minimax = LiteLlm(
    model="vertex_ai/minimaxai/minimax-m2-maas",
    vertex_project="deltorobarba",
    vertex_location="global")

# GLM 5 - https://docs.cloud.google.com/vertex-ai/generative-ai/docs/maas/zaiorg/glm-5
glm = LiteLlm(
    model="vertex_ai/zai-org/glm-5-maas",
    vertex_project="deltorobarba",
    vertex_location="global")

# ⚠️ ----- Quick Test on Model Availability -----
deepseek = "vertex_ai/deepseek-ai/deepseek-v3.2-maas"     # ✅
kimi = "vertex_ai/moonshotai/kimi-k2-thinking-maas"       # ✅
qwen = "vertex_ai/qwen/qwen3-next-80b-a3b-thinking-maas"  # ✅
minimax = "vertex_ai/minimaxai/minimax-m2-maas"           # ✅
glm ="vertex_ai/zai-org/glm-5-maas"                       # ✅

project = "deltorobarba"
location = "global"
question = "Hi"

try:
    response = litellm.completion(
        model=deepseek,                                    # <--- ⚠️ Add model here
        messages=[{"role": "user", "content": question}],
        temperature=0.2,
        #max_tokens=264,
        vertex_project=project,
        vertex_location=location)
    print("✅ Success: The model is reachable!")
    print(response.choices[0].message.content)
except Exception as e:
    print(f"❌ Failed: Connectivity or Model Name error: {e}")

✅ Success: The model is reachable!
你好！😊 很高兴见到你！

我是DeepSeek，一个热心的AI助手。无论你想聊天、寻求帮助、解答问题，还是需要创作灵感，我都很乐意为你提供支持！

有什么我可以帮助你的吗？不管是学习、工作、生活上的问题，还是只是想随便聊聊，我都在这里等着你呢～ ✨


In [9]:
# @title Vertex AI - Deployment Recommendation
#################################################

from vertexai import model_garden

# Model Garden Identifier (Format: google/gemma4@gemma-4-26b-a4b-it oder google/gemma4@gemma-4-31b-it)
MODEL_ID = "google/gemma4@gemma-4-26b-a4b-it"

# Model Garden Objekt initialisieren
model = model_garden.OpenModel(MODEL_ID)

# Empfohlene/verifizierte Deployment-Optionen abrufen (concise=True für lesbare Übersicht)
deploy_options = model.list_deploy_options(concise=True)
print(deploy_options)

[Option 1: vLLM 256K context]
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:gemma4",
    machine_type="g4-standard-48",
    accelerator_type="NVIDIA_RTX_PRO_6000",
    accelerator_count=1,

[Option 2: vLLM 256K context spec decode]
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:20260505_0916_RC01",
    machine_type="g4-standard-48",
    accelerator_type="NVIDIA_RTX_PRO_6000",
    accelerator_count=1,

[Option 3: vLLM 256K context]
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:gemma4",
    machine_type="g4-standard-96",
    accelerator_type="NVIDIA_RTX_PRO_6000",
    accelerator_count=2,

[Option 4: vLLM 256K context]
    serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:gemma4",
    machine_type="a3-highgpu-1g",
    acceler

In [ ]:
# @title Vertex AI - Deployment Script

"""
Recommendation: NVIDIA RTX Pro 6000 (Blackwell) with VRAM	96 GB
We prefer NVIDIA L4 (Ada Lovelace) with VRAM 24 GB. Capacity: The L4 has 24 GB of GDDR6 memory.
The Problem: The Gemma 4 26B model in full precision (BF16) requires roughly 52 GB of VRAM just to load the weights. It will not fit on a single L4 in that state.
The Solution (Quantization): To run this on an L4, you must use 4-bit quantization (like AWQ or INT4). In 4-bit mode, the weights take up only ~15–18 GB, which fits comfortably on a single L4.
The Catch: You will likely need to reduce your context window significantly (e.g., from 256K down to 8K or 16K) to stay within the L4's 24GB limit.
"""


"""
import vertexai
from vertexai import model_garden

vertexai.init(project="lunar-352813", location="europe-west4")

model = model_garden.OpenModel("google/gemma4@gemma-4-26b-a4b-it")
endpoint = model.deploy(
  accept_eula=True,
  machine_type="g4-standard-48",
  accelerator_type="NVIDIA_RTX_PRO_6000",
  accelerator_count=1,
  serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:gemma4",
  endpoint_display_name="gemma-4-26b-a4b-it-mg-one-click-deploy",
  model_display_name="gemma-4-26b-a4b-it-1777568163447",
  use_dedicated_endpoint=True,
  reservation_affinity_type="NO_RESERVATION",
)
"""

import vertexai
from vertexai import model_garden

vertexai.init(project="lunar-352813", location="europe-west4")

model = model_garden.OpenModel("google/gemma4@gemma-4-31b-it")
endpoint = model.deploy(
  accept_eula=True,
  machine_type="g4-standard-48",
  accelerator_type="NVIDIA_RTX_PRO_6000",
  accelerator_count=1,
  serving_container_image_uri="us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/pytorch-vllm-serve:gemma4",
  endpoint_display_name="gemma-4-31b-it-mg-one-click-deploy",
  model_display_name="gemma-4-31b-it-1777570278133",
  use_dedicated_endpoint=True,
  reservation_affinity_type="NO_RESERVATION",
)